# 프롬프트 다양화 실험

## 아이디어
지금은 32개 샘플이 **전부 같은 시스템 프롬프트**에서 나옵니다.
다양성이 `temperature` 하나에만 의존하고 있어요.

프롬프트를 4종으로 나누면(각 8샘플씩) **서로 다른 방식으로 접근**하게 됩니다.

```
기존:  프롬프트 A × 32샘플
변경:  프롬프트 A × 8  +  B × 8  +  C × 8  +  D × 8
```

## 왜 다수결에 유리한가
> **오답은 흩어질수록, 정답은 모일수록 좋다.**

정답은 어떤 프롬프트로 풀든 같은 값입니다. 반면 오답은 접근법이 다르면 **다른 값으로** 틀립니다.
같은 프롬프트로 32번 뽑으면 **같은 실수를 32번 반복**할 수 있어요 — RFT 실험에서 "오답이 체계적"이라는 걸 확인했죠.

## 기대되는 또 하나
**`pass@32`가 오를 수 있습니다.** 다른 프롬프트가 다른 풀이 경로를 탐색하므로,
기존에 32번 뽑아도 못 찾던 정답을 찾아낼 가능성이 있어요.
RFT는 천장(0.8633) 아래에서만 움직였지만, 이건 **천장 자체를 올릴 수도** 있습니다.

## 기준선
| 지표 | 값 |
|---|---|
| maj@8 (프롬프트 1종) | 0.7167 |
| **maj@32 (프롬프트 1종)** | **0.7433** |
| **pass@32** | **0.8633** |
| 평균 득표율 | 0.726 |

## 실행 순서
`[1]` `[2]` `[3]` → ⛔Restart → `[1]` → `[4]`~`[9]`  (생성 약 52분)


---
## [1] 설정 ▶️

In [ ]:
N_SAMPLES  = 32        # 총 샘플 수 (프롬프트 종류로 나눠 배분)
TEMP       = 0.8
MAX_TOKENS = 1024
VALID_N    = 300       # 절대 변경 금지 — 기준선과 같은 300문제
SEED       = 42        # 절대 변경 금지
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
print(f"{VALID_N}문제 x {N_SAMPLES}샘플")

---
## [2] vLLM 설치 ⏭️

In [ ]:
!pip install -q -U vllm 2>&1 | tail -3

---
## [3] protobuf ⏭️
---
## ⛔ Restart Session → [1]부터
---

In [ ]:
!pip install -q -U "protobuf>=6.33.6,<7" 2>&1 | tail -2
import google.protobuf as p
print("protobuf", p.__version__); assert p.__version__.startswith("6.")

---
## [4] 데이터 ▶️ 기준선과 같은 300문제

In [ ]:
import glob, os, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p

TRAIN_PATH = find_csv(["train"], must_not=["filtered","ids","leaderboard","test"])
BAD_PATH   = find_csv(["filtered","ids"])
train = pd.read_csv(TRAIN_PATH)
train = train[~train["id"].isin(set(pd.read_csv(BAD_PATH)["id"]))].reset_index(drop=True)
assert len(train) == 16373

work = train.sample(VALID_N, random_state=SEED).reset_index(drop=True)
gold = work["answer"].tolist()
print(f"{len(work)}문제 | 첫 id: {work.iloc[0]['id']}  (train-004925 여야 함)")

---
## [5] 답 추출기 ▶️

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]

_c=[(r"\boxed{132}",132),(r"\boxed{-2,025,078}",-2025078),(r"\boxed{\frac{7}{2}}",None)]
print("parser FAILURES:", sum(parse_answer(t)!=w for t,w in _c), "/", len(_c))

---
## [6] 프롬프트 4종 정의 ▶️

### 설계 원칙
단순히 말을 바꾼 게 아니라 **접근 방식 자체가 다르게** 만들었습니다.
말만 바꾸면 모델이 비슷하게 풀어서 다양성이 안 생깁니다.

| # | 전략 | 노리는 것 |
|---|---|---|
| **A** | 기존 (간결·단계별) | 기준선. 검증된 프롬프트 |
| **B** | 검산 강조 | 산술 실수 감소 |
| **C** | 계획 먼저 | 접근법 자체를 다르게 |
| **D** | 모든 중간값 명시 | 긴 풀이. 생략에서 오는 실수 방지 |

### 구조
문제 하나당 프롬프트 4종 × 8샘플 = 32샘플.
vLLM에는 `300문제 × 4종 = 1,200개` 프롬프트를 넣고 각각 `n=8`로 생성한 뒤,
나중에 문제별로 다시 묶습니다.

In [ ]:
from transformers import AutoTokenizer

BOX = " End your response with the final integer inside \\boxed{}."

SYSTEMS = [
    # A — 기존 (기준선)
    "You are an expert competition mathematician. Solve the problem step by step, "
    "concisely. The final answer is ALWAYS a single integer." + BOX,

    # B — 검산 강조
    "You are a careful mathematician. Solve the problem, then verify every arithmetic "
    "step before committing to an answer. If a check fails, redo that step. "
    "The final answer is ALWAYS a single integer." + BOX,

    # C — 계획 먼저
    "First restate what is given and what is asked. Then choose a strategy and state it. "
    "Only then carry out the computation. The final answer is ALWAYS a single integer." + BOX,

    # D — 중간값 전부 명시
    "Solve the problem showing every intermediate value explicitly. Never skip a "
    "calculation or do arithmetic mentally. The final answer is ALWAYS a single integer." + BOX,
]
LABELS  = ["A_baseline", "B_verify", "C_plan", "D_explicit"]
N_VAR   = len(SYSTEMS)
N_PER   = N_SAMPLES // N_VAR
assert N_SAMPLES % N_VAR == 0, "N_SAMPLES가 프롬프트 종류 수로 나눠떨어져야 합니다"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
prompts = []
for q in work["question"]:
    for s in SYSTEMS:
        prompts.append(tok.apply_chat_template(
            [{"role":"system","content":s},{"role":"user","content":q}],
            tokenize=False, add_generation_prompt=True))

print(f"프롬프트 {N_VAR}종 x {N_PER}샘플 = 문제당 {N_SAMPLES}샘플")
print(f"vLLM 요청 수: {len(prompts)}개 (각 n={N_PER})")

---
## [7] 생성 ▶️ 약 52분

In [ ]:
import time
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, dtype="half", max_model_len=4096,
          gpu_memory_utilization=0.90, tensor_parallel_size=1,
          seed=SEED, trust_remote_code=True)

sp = SamplingParams(n=N_PER, temperature=TEMP, top_p=0.95,
                    max_tokens=MAX_TOKENS, seed=SEED)

t0 = time.time()
outs = llm.generate(prompts, sp)
print(f"\n생성 {(time.time()-t0)/60:.1f}분")

---
## [8] 프롬프트별 성능 ▶️ (GPU 미사용)

**어떤 프롬프트가 좋고 나쁜지** 따로 봅니다.
하나가 크게 나쁘면 그건 빼는 게 낫습니다 — 32샘플 중 8개를 노이즈로 채우는 셈이니까요.

각 프롬프트의 `maj@8`을 기준선 **0.7167**과 비교하세요.

In [ ]:
import numpy as np
from collections import Counter

# outs를 문제별 x 프롬프트별로 재구성
per_var = {lab: [] for lab in LABELS}          # 프롬프트별 답 후보
pooled  = []                                   # 문제별 전체 32개
for i in range(len(work)):
    allv = []
    for j, lab in enumerate(LABELS):
        vals = [parse_answer(c.text) for c in outs[i*N_VAR + j].outputs]
        per_var[lab].append(vals)
        allv += vals
    pooled.append(allv)

def maj(vals, fb=0):
    v = [x for x in vals if x is not None]
    return Counter(v).most_common(1)[0][0] if v else fb

print(f"{'프롬프트':<14} {'maj@'+str(N_PER):>8} {'pass@'+str(N_PER):>9} {'파싱실패':>8}")
print("-" * 44)
for lab in LABELS:
    m = np.mean([int(maj(v)) == int(g) for v, g in zip(per_var[lab], gold)])
    p = np.mean([any(x is not None and int(x) == int(g) for x in v)
                 for v, g in zip(per_var[lab], gold)])
    f = np.mean([x is None for v in per_var[lab] for x in v])
    print(f"{lab:<14} {m:>8.4f} {p:>9.4f} {f:>7.2%}")
print(f"\n기준선 maj@8 = 0.7167 (프롬프트 A만 8샘플)")

---
## [9] 통합 결과 + 최적 조합 탐색 ▶️

**① 4종 전부 합친 maj@32** — 기준선 0.7433과 직접 비교

**② 부분집합 탐색** — 나쁜 프롬프트를 빼면 더 나아지는지.
이미 생성한 결과를 다시 묶기만 하는 거라 **GPU를 전혀 안 씁니다.**
단, 샘플 수가 줄어드는 조합(3종=24샘플)은 그만큼 불리하니 감안해서 보세요.

In [ ]:
from itertools import combinations

def score(list_of_vals, label, n):
    m = np.mean([int(maj(v)) == int(g) for v, g in zip(list_of_vals, gold)])
    p = np.mean([any(x is not None and int(x) == int(g) for x in v)
                 for v, g in zip(list_of_vals, gold)])
    sh = np.mean([Counter([x for x in v if x is not None]).most_common(1)[0][1]/len(v)
                  if any(x is not None for x in v) else 0 for v in list_of_vals])
    print(f"{label:<26} n={n:>2}  maj={m:.4f}  pass={p:.4f}  득표율={sh:.3f}")
    return m

print("=" * 68)
print("기준선 (프롬프트 A 1종)       n=32  maj=0.7433  pass=0.8633  득표율=0.726")
print("=" * 68)
full = score(pooled, "4종 전부", N_SAMPLES)

print("\n--- 부분집합 (샘플 수가 줄어드는 점 감안) ---")
for r in [3, 2]:
    for combo in combinations(range(N_VAR), r):
        sub = [sum((per_var[LABELS[j]][i] for j in combo), []) for i in range(len(work))]
        score(sub, "+".join(LABELS[j][0] for j in combo), r * N_PER)

print("\n" + "=" * 68)
print(f"4종 전부 maj@32 = {full:.4f}   기준선 0.7433 대비 {(full-0.7433)*100:+.2f}%p")
print("리더보드 환산(x0.63):", f"{(full-0.7433)*0.63*100:+.2f}%p  → 예상 {0.78580+(full-0.7433)*0.63:.5f}")